# Build your own animal for morphing_birds

Use this notebook before the DMD workshop notebook when you have custom keypoint data shaped `(n_frames, n_markers, 3)`. The aim is to define the marker order, the display polygons, the analysis markers, and a first `Animal3D` object that animates correctly.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    "uv",
    "pip",
    "install",
    "--python",
    sys.executable,
    "git+https://github.com/LydiaFrance/morphing_birds.git@custom-skeleton",
])


In [ ]:
import numpy as np
import plotly.io as pio

from morphing_birds import Animal3D, SkeletonDefinition, animate_plotly

if "google.colab" in sys.modules:
    pio.renderers.default = "colab"

np.set_printoptions(precision=3, suppress=True)


## 1. Start with data shaped `(frames, markers, xyz)`

Replace this synthetic example with your own loaded array. The important rule is that axis 1 must match `marker_names` exactly.


In [ ]:
marker_names = [
    "nose",
    "neck",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_front_foot",
    "right_front_foot",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_back_foot",
    "right_back_foot",
    "tail_base",
    "tail_tip",
    "tracking_tag",
]

# A small top-down example: x is left/right, y is nose/tail, z is height.
rest_pose = np.array(
    [
        [0.00, 1.35, 0.00],   # nose
        [0.00, 0.95, 0.00],   # neck
        [-0.35, 0.70, 0.00],  # left_shoulder
        [0.35, 0.70, 0.00],   # right_shoulder
        [-0.60, 0.35, 0.00],  # left_elbow
        [0.60, 0.35, 0.00],   # right_elbow
        [-0.80, 0.05, 0.00],  # left_front_foot
        [0.80, 0.05, 0.00],   # right_front_foot
        [-0.30, -0.45, 0.00], # left_hip
        [0.30, -0.45, 0.00],  # right_hip
        [-0.52, -0.75, 0.00], # left_knee
        [0.52, -0.75, 0.00],  # right_knee
        [-0.70, -1.05, 0.00], # left_back_foot
        [0.70, -1.05, 0.00],  # right_back_foot
        [0.00, -0.70, 0.00],  # tail_base
        [0.00, -1.35, 0.00],  # tail_tip
        [0.00, 0.15, 0.25],   # tracking_tag
    ],
    dtype=float,
)

n_frames = 60
phase = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
motion = np.repeat(rest_pose[None, :, :], n_frames, axis=0)

left_step = np.sin(phase)
right_step = np.sin(phase + np.pi)
motion[:, marker_names.index("left_front_foot"), 2] += 0.18 * np.maximum(left_step, 0)
motion[:, marker_names.index("right_back_foot"), 2] += 0.14 * np.maximum(left_step, 0)
motion[:, marker_names.index("right_front_foot"), 2] += 0.18 * np.maximum(right_step, 0)
motion[:, marker_names.index("left_back_foot"), 2] += 0.14 * np.maximum(right_step, 0)
motion[:, marker_names.index("tail_tip"), 0] += 0.08 * np.sin(phase)
motion[:, marker_names.index("neck"), 2] += 0.03 * np.sin(2 * phase)

assert motion.shape == (n_frames, len(marker_names), 3)
motion.shape


## 2. Define the skeleton

`body_sections` are polygons or line strips for display. `analysis_exclude` markers are displayed but skipped by PCA/SVD/FFT/DMD, so only put fixed landmarks there. Left/right pairs and centre markers make symmetry helpers explicit even when your names do not use the builtin bird conventions.


In [ ]:
body_sections = {
    "head": ["right_shoulder", "nose", "left_shoulder"],
    "torso": ["right_shoulder", "right_hip", "tail_base", "left_hip", "left_shoulder"],
    "tail": ["tail_base", "tail_tip"],
    "left_foreleg": ["left_shoulder", "left_elbow", "left_front_foot", "left_elbow", "left_shoulder"],
    "right_foreleg": ["right_shoulder", "right_elbow", "right_front_foot", "right_elbow", "right_shoulder"],
    "left_hindleg": ["left_hip", "left_knee", "left_back_foot", "left_knee", "left_hip"],
    "right_hindleg": ["right_hip", "right_knee", "right_back_foot", "right_knee", "right_hip"],
}

marker_pairs = [
    ("left_shoulder", "right_shoulder"),
    ("left_elbow", "right_elbow"),
    ("left_front_foot", "right_front_foot"),
    ("left_hip", "right_hip"),
    ("left_knee", "right_knee"),
    ("left_back_foot", "right_back_foot"),
]

centre_markers = ["nose", "neck", "tail_base", "tail_tip", "tracking_tag"]

skel = SkeletonDefinition.from_markers(
    "workshop_animal",
    marker_names,
    body_sections=body_sections,
    analysis_exclude=["tracking_tag"],
    marker_pairs=marker_pairs,
    centre_markers=centre_markers,
)

print("All markers:", skel.all_marker_names)
print("Analysis markers:", skel.analysis_markers)
print("Display-only markers:", skel.display_only_markers)


## 3. Build the animal and check the raw animation

`rest_pose` is the static shape used for display-only markers and for the first view of the animal.


In [ ]:
rest_pose = np.nanmean(motion, axis=0)
animal = Animal3D(skel, data=rest_pose)

fig = animate_plotly(animal, motion, axes_visible=False)
fig.show()


## 4. Handoff to the DMD notebook

The maths should use only the analysis subset. Reconstructed analysis data can be animated directly against the same `animal`.


In [ ]:
X = animal.get_analysis_data(motion)
X_flat = X.reshape(X.shape[0], -1)

# In the DMD notebook, replace this with your reconstructed flat array.
recon_flat = X_flat.copy()
recon = recon_flat.reshape(-1, X.shape[1], 3)

fig = animate_plotly(animal, recon, axes_visible=False)
fig.show()

print("DMD input:", X_flat.shape)
print("Animation reconstruction:", recon.shape)


## Copy this cell into the main DMD workshop notebook

Keep `marker_names`, `body_sections`, `marker_pairs`, `centre_markers`, `skel`, `rest_pose`, `animal`, and `motion` together. Replace only the synthetic `motion` block with your data loader.


In [ ]:
marker_names = [
    "nose",
    "neck",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_front_foot",
    "right_front_foot",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_back_foot",
    "right_back_foot",
    "tail_base",
    "tail_tip",
    "tracking_tag",
]

# Replace this synthetic motion with your data loader.
rest_pose = np.array(
    [
        [0.00, 1.35, 0.00],
        [0.00, 0.95, 0.00],
        [-0.35, 0.70, 0.00],
        [0.35, 0.70, 0.00],
        [-0.60, 0.35, 0.00],
        [0.60, 0.35, 0.00],
        [-0.80, 0.05, 0.00],
        [0.80, 0.05, 0.00],
        [-0.30, -0.45, 0.00],
        [0.30, -0.45, 0.00],
        [-0.52, -0.75, 0.00],
        [0.52, -0.75, 0.00],
        [-0.70, -1.05, 0.00],
        [0.70, -1.05, 0.00],
        [0.00, -0.70, 0.00],
        [0.00, -1.35, 0.00],
        [0.00, 0.15, 0.25],
    ],
    dtype=float,
)
motion = np.repeat(rest_pose[None, :, :], 60, axis=0)
assert motion.ndim == 3 and motion.shape[1:] == (len(marker_names), 3)

body_sections = {
    "head": ["right_shoulder", "nose", "left_shoulder"],
    "torso": ["right_shoulder", "right_hip", "tail_base", "left_hip", "left_shoulder"],
    "tail": ["tail_base", "tail_tip"],
    "left_foreleg": ["left_shoulder", "left_elbow", "left_front_foot", "left_elbow", "left_shoulder"],
    "right_foreleg": ["right_shoulder", "right_elbow", "right_front_foot", "right_elbow", "right_shoulder"],
    "left_hindleg": ["left_hip", "left_knee", "left_back_foot", "left_knee", "left_hip"],
    "right_hindleg": ["right_hip", "right_knee", "right_back_foot", "right_knee", "right_hip"],
}
marker_pairs = [
    ("left_shoulder", "right_shoulder"),
    ("left_elbow", "right_elbow"),
    ("left_front_foot", "right_front_foot"),
    ("left_hip", "right_hip"),
    ("left_knee", "right_knee"),
    ("left_back_foot", "right_back_foot"),
]
centre_markers = ["nose", "neck", "tail_base", "tail_tip", "tracking_tag"]

skel = SkeletonDefinition.from_markers(
    "workshop_animal",
    marker_names,
    body_sections=body_sections,
    analysis_exclude=["tracking_tag"],
    marker_pairs=marker_pairs,
    centre_markers=centre_markers,
)
rest_pose = np.nanmean(motion, axis=0)
animal = Animal3D(skel, data=rest_pose)

X = animal.get_analysis_data(motion)
X_flat = X.reshape(X.shape[0], -1)


## Troubleshooting

- Marker order mismatch: `marker_names[i]` must name `motion[:, i, :]`.
- Wrong units or axes: check whether your data is metres, centimetres, pixels, or whether vertical is `z`.
- Missing values: use `NaN` for unknown coordinates so `np.nanmean` does the right thing.
- Polygon marker typo: every name in `body_sections` must also appear in `marker_names`.
- Display-only markers: put fixed landmarks in `analysis_exclude`; moving keypoints usually belong in the analysis set.
